---
title: "SPARQL Class Instance Counts"
format:
  html:
    code-fold: true
execute:
  enabled: false
---

This notebook demonstrates basic SPARQL queries against the ClimateKG Wikibase instance to count how many items belong to each major class in the corpus hierarchy.

**Wikibase Instance**: [ClimateKG Production](https://prod-climatekg.semanticclimate.org/wiki/Main_Page)

**SPARQL Endpoint**: https://prod-climatekg.semanticclimate.org/bigdata/sparql

**📓 Download Notebook**: [class-counts.ipynb](https://github.com/TIBHannover/Climate-KG-data/blob/main/sparql_queries/class-counts.ipynb) - Run this notebook yourself to execute live queries against ClimateKG.

---

## Setup

First, we'll import the necessary libraries and configure the SPARQL endpoint.

In [11]:
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd
from IPython.display import display, Markdown, HTML

# Configure the SPARQL endpoint
SPARQL_ENDPOINT = "https://prod-climatekg.semanticclimate.org/query/proxy/sparql"
WIKIBASE_URL = "https://prod-climatekg.semanticclimate.org"

print("✓ Libraries imported successfully")
print(f"✓ SPARQL Endpoint: {SPARQL_ENDPOINT}")

✓ Libraries imported successfully
✓ SPARQL Endpoint: https://prod-climatekg.semanticclimate.org/query/proxy/sparql


## Helper Functions

We'll create helper functions to execute SPARQL queries and display results.

In [12]:
def count_class_instances(class_qid, class_name=""):
    """
    Count the number of items that are instances of a given class.
    Uses P1 (instance of) to find items belonging to the given class QID.
    """
    # Build the SPARQL query using P1 (instance of)
    query = f"""PREFIX wd: <{WIKIBASE_URL}/entity/>
PREFIX wdt: <{WIKIBASE_URL}/prop/direct/>

SELECT (COUNT(?item) AS ?count) WHERE {{
  ?item wdt:P1 wd:{class_qid} .
}}"""

    # Execute the query
    sparql = SPARQLWrapper(SPARQL_ENDPOINT)
    sparql.setQuery(query)
    sparql.setReturnFormat(JSON)

    try:
        results = sparql.query().convert()
        count = int(results["results"]["bindings"][0]["count"]["value"])

        # Generate query URL for Wikibase SPARQL interface
        import urllib.parse
        query_url = f"{WIKIBASE_URL}/query/#" + urllib.parse.quote(query)

        return {
            "class_qid": class_qid,
            "class_name": class_name,
            "count": count,
            "query": query,
            "query_url": query_url,
            "entity_url": f"{WIKIBASE_URL}/wiki/Item:{class_qid}"
        }
    except Exception as e:
        print(f"Error querying {class_qid}: {e}")
        return {
            "class_qid": class_qid,
            "class_name": class_name,
            "count": 0,
            "query": query,
            "error": str(e)
        }

def display_count_result(result):
    """
    Display the count result in a formatted way.
    """
    class_label = f"{result['class_name']} ({result['class_qid']})" if result['class_name'] else result['class_qid']

    display(Markdown(f"### {class_label}"))

    if 'error' in result:
        display(HTML(f'<p style="color: red;">&#10060; Error: {result["error"]}</p>'))
        return

    # Display count
    display(HTML(f'<p style="font-size: 1.5em; font-weight: bold; color: #2c3e50;">Count: {result["count"]:,} items</p>'))

    # Display links
    display(HTML(f'''
    <p>
        <a href="{result['entity_url']}" target="_blank" style="margin-right: 15px;">View Class in Wikibase</a>
        <a href="{result['query_url']}" target="_blank">Run Query in SPARQL Interface</a>
    </p>
    '''))

    # Display SPARQL query
    display(Markdown("**SPARQL Query:**"))
    display(HTML(f'<pre style="background-color: #f4f4f4; padding: 10px; border-radius: 5px; overflow-x: auto;"><code>{result["query"]}</code></pre>'))

    display(HTML('<hr style="margin: 30px 0;">'))

print("✓ Helper functions defined")

✓ Helper functions defined


---

## Class Instance Counts

The following sections show the count of items for each major class in the corpus hierarchy. Each count is accompanied by:

- The SPARQL query used to obtain the count
- A link to view the class definition in Wikibase
- A link to run the query in the Wikibase SPARQL interface

### Corpus Hierarchy Classes

The classes are counted in the following order, representing the typical hierarchy of IPCC corpus items:

- **Q2**: Work (abstract creation — the base class for all works)
- **Q3**: Report Series (a series of individual monograph standalone works)
- **Q4**: Report (a standalone written work — book/monograph/volume)
- **Q5**: Text Division (manuscript hierarchical component)
- **Q1**: Category (subject or topic tag)
- **Q2087**: Acronym (IPCC AR6 acronym)
- **Q3998**: Author (IPCC AR6 author — person)

---

### Q2 - Work

In [13]:
result_q2 = count_class_instances("Q2", "Work")
display_count_result(result_q2)

### Work (Q2)

**SPARQL Query:**

### Q3 - Report Series

In [14]:
result_q3 = count_class_instances("Q3", "Report Series")
display_count_result(result_q3)

### Report Series (Q3)

**SPARQL Query:**

### Q4 - Report

In [15]:
result_q4 = count_class_instances("Q4", "Report")
display_count_result(result_q4)

### Report (Q4)

**SPARQL Query:**

### Q5 - Text Division

In [16]:
result_q5 = count_class_instances("Q5", "Text Division")
display_count_result(result_q5)

### Text Division (Q5)

**SPARQL Query:**

### Q1 - Category

In [17]:
result_q1 = count_class_instances("Q1", "Category")
display_count_result(result_q1)

### Category (Q1)

**SPARQL Query:**

### Q2087 - Acronym

In [18]:
result_q2087 = count_class_instances("Q2087", "Acronym")
display_count_result(result_q2087)

### Acronym (Q2087)

**SPARQL Query:**

### Q3998 - Author

In [19]:
result_q3998 = count_class_instances("Q3998", "Author")
display_count_result(result_q3998)

### Author (Q3998)

**SPARQL Query:**

---

## Summary Table

Here's a summary of all the class counts:

In [20]:
# Collect all results
all_results = [result_q2, result_q3, result_q4, result_q5, result_q1, result_q2087, result_q3998]

# Create summary DataFrame
summary_data = []
for result in all_results:
    if 'error' not in result:
        summary_data.append({
            'Class': result['class_qid'],
            'Name': result['class_name'],
            'Count': result['count']
        })

summary_df = pd.DataFrame(summary_data, columns=['Class', 'Name', 'Count'])

# Display as formatted table
display(Markdown("### Instance Counts by Class"))
if not summary_df.empty:
    display(summary_df.style.format({'Count': '{:,}'}).set_properties(**{
        'text-align': 'left',
        'font-size': '1.1em'
    }))
    print(f"\n✓ Total items counted: {summary_df['Count'].sum():,}")

else:    print("No results available — run the query cells above first.")

### Instance Counts by Class

,Class,Name,Count
0,Q2,Work,0
1,Q3,Report Series,1
2,Q4,Report,7
3,Q5,Text Division,10
4,Q1,Category,"1,001"
5,Q2087,Acronym,"1,910"
6,Q3998,Author,932



✓ Total items counted: 3,861


---

## Understanding the Queries

### SPARQL Query Structure

All the queries above follow the same basic pattern:

```sparql
PREFIX wd: <https://prod-climatekg.semanticclimate.org/entity/>
PREFIX wdt: <https://prod-climatekg.semanticclimate.org/prop/direct/>

SELECT (COUNT(?item) AS ?count) WHERE {
  ?item wdt:P1 wd:QX .
}
```

**Explanation:**
- `PREFIX wd:` - Defines the namespace for entities (classes and items)
- `PREFIX wdt:` - Defines the namespace for direct properties
- `?item wdt:P1 wd:QX` - Finds all items (`?item`) that have property P1 (instance of) pointing to class QX
- `COUNT(?item)` - Counts the number of matching items

### Key Property

- **P1**: "instance of" — This property links items to their class in ClimateKG

### Running Your Own Queries

You can run these queries directly in the [ClimateKG SPARQL Query Service](https://prod-climatekg.semanticclimate.org/query/) or use the links provided above for each class.

---

## Next Steps

This notebook demonstrates basic instance counting. You can extend these queries to:

- Filter by additional properties
- Retrieve actual item labels and data
- Perform aggregations and statistics
- Explore relationships between classes

**Related Notebooks:**
- [View this notebook on GitHub](https://github.com/TIBHannover/Climate-KG-data/blob/main/sparql_queries/class-counts.ipynb)

**External Resources:**
- [ClimateKG Wikibase](https://prod-climatekg.semanticclimate.org/wiki/Main_Page)
- [SPARQL Query Service](https://prod-climatekg.semanticclimate.org/query/)
- [Wikibase SPARQL Documentation](https://www.mediawiki.org/wiki/Wikibase/Indexing/SPARQL_Query_Examples)